# 02: Quantification of Growth-Induced Pressure (GIP)

## Overview
This notebook applies the established calibration model to timelapse imagery of bacterial growth. It quantifies how E. coli proliferation inside micro-chambers generates mechanical pressure over time.

### Key Features:
- Spatial Masking: Crops images to the central chamber region to minimize boundary effects.
- Temporal Analysis: Maps wall separation at 30-minute intervals to the calibrated pressure scale.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.measure import label, regionprops
from natsort import natsorted

# 1. Load Calibration Model
base_dir = os.path.dirname(os.getcwd())
calibration_file = os.path.join(base_dir, "data", "results", "Calibration_Model.csv")
growth_folder = os.path.join(base_dir, "data", "raw", "growth")

if os.path.exists(calibration_file):
    cal_df = pd.read_csv(calibration_file)
    # Establish baseline distance (0 mbar)
    baseline_dist = cal_df[cal_df['Pressure_mbar'] == 0]['Distance_um'].values[0]

# 2. Analyze Growth Timelapse
time_results = []
if os.path.exists(growth_folder):
    files = natsorted([f for f in os.listdir(growth_folder) if f.endswith('.png')])
    
    for i, file in enumerate(files):
        img = imread(os.path.join(growth_folder, file))
        # Crop to middle region (y: 100-300) for uniform deformation analysis
        cropped = img[100:300, :]
        labeled = label(cropped > 0)
        props = [p for p in regionprops(labeled) if p.area > 700]
        
        if len(props) >= 2:
            # Calculate current distance (simplified centroid x-dist for growth monitoring)
            dist = abs(props[0].centroid[1] - props[1].centroid[1]) * 0.06587
            time_results.append({"Time_hours": i * 0.5, "Distance_um": dist})

df_growth = pd.DataFrame(time_results)

# 3. Visualization
plt.figure(figsize=(10,6))
plt.plot(df_growth['Time_hours'], df_growth['Distance_um'], 'o-', color='crimson')
plt.title("Bacterial Growth-Induced Wall Deformation Over Time")
plt.xlabel("Time (Hours)")
plt.ylabel("Wall Separation (µm)")
plt.grid(True, alpha=0.3)
plt.show()